# L7-L10：学习理论

## 依照 L7-8、L9 与 L10 编写的完整中文讲义

本讲义依照以下三份课程幻灯片（slides）的可见内容与顺序编写：

- [L7-8：Learning Theory](../slides/L7-8.pdf)，38 页，2026 年 5 月 5 日。
- [L9：Learning Theory - 3](../slides/L9.pdf)，24 页，2026 年 5 月 12 日。
- [L10：Learning Theory - 4](../slides/L10.pdf)，27 页，2026 年 5 月 12 日。

三份标题页都注明本课程为 **CIT413048：Mathematical Foundations of ML**，授课教师为慕尼黑工业大学（TU München）数学系的 **Suvrit Sra**。

三份幻灯片构成本讲义的主线。标记为 **补充（Supplement）** 的内容来自此前的综合笔记、Francis Bach 的 *Learning Theory from First Principles*，以及 Shalev-Shwartz 和 Ben-David 的 *Understanding Machine Learning* 配套解答。幻灯片中因动画逐步显现而重复的页面已合并，但其最终内容没有删减。所有数学公式均使用标准 Markdown（standard Markdown）的 `$...$` 与 `$$...$$` 定界符。

课程的中心逻辑链是

$$
\text{经验风险最小化（empirical risk minimization, ERM）}
\longrightarrow \text{泛化（generalization）}
\longrightarrow \text{概率近似正确学习（probably approximately correct learning, PAC learning）}
\longrightarrow \text{一致收敛（uniform convergence）}
\longrightarrow \text{增长函数（growth function）与 VC 维（Vapnik--Chervonenkis dimension, VC dimension）}
\longrightarrow \text{统计学习基本定理（fundamental theorem of statistical learning）}
\longrightarrow \text{算法稳定性（algorithmic stability）}.
$$


# 第一部分——L7-8：从 ERM 到 PAC 与一致收敛


# 1. 课程进行到哪里，以及为什么要研究学习理论

*幻灯片主线：L7-8，第 1--3 页。*

在 L7-8 之前，课程已经学习了：

1. 分类（classification）与经验风险最小化的思想；
2. 过拟合（overfitting）、归纳偏置（inductive bias）与假设类（hypothesis class）；
3. 线性分类器（linear classifier）、逻辑回归（logistic regression）与支持向量机（support vector machine, SVM）；
4. 线性回归（linear regression）与普通最小二乘法（ordinary least squares, OLS）；
5. 非线性分类器（nonlinear classifier）与非线性特征（nonlinear feature）；
6. 核方法（kernel method）、相关理论与表示定理（representer theorem）。

L7-8 提出了一个更根本的问题：**从数据中学习究竟是什么意思？** 研究学习理论（learning theory）是为了

- 通过归纳偏置、隐式正则化（implicit regularization）与结构发现更好的模型；
- 发现能在统计权衡（statistical tradeoff）与计算权衡（computational tradeoff）之间作出适当选择的算法；
- 从科学上理解并解释学习过程为什么成功或失败。

目标是从有限的训练集（training set）中学习，并在未见数据（unseen data）上准确预测。第 3 页上小型的“Learn / ML”书本图形，表达了从仅仅使用机器学习（machine learning, ML）转向理解“学习本身”的过程。


# 2. 统计设定：数据、假设、损失与风险

*幻灯片主线：L7-8，第 4--6 页。*

本讲采用监督学习（supervised learning）的标准统计设定。

设数据域（data domain）为任意集合 $\mathcal X$，通常是 $\mathbb R^d$；标签域（label domain）为离散集合 $\mathcal Y$，例如 $\{0,1\}$ 或 $\{-1,1\}$。若要强调输入的表示（representation），可以把特征映射（feature map）写为 $\phi(x)$。

令

$$
\mathcal Z=\mathcal X\times\mathcal Y,
\qquad
Z=(X,Y)\sim\mathbb P,
$$

其中 $\mathbb P$ 是固定但未知的数据分布（data distribution）。训练样本（training sample）为

$$
S=(Z_1,\ldots,Z_n)
=((X_1,Y_1),\ldots,(X_n,Y_n))
\sim\mathbb P^n.
$$

缩写 i.i.d. 表示独立同分布（independent and identically distributed）。一个假设（hypothesis）、预测规则（prediction rule）或分类器是映射

$$
h:\mathcal X\to\mathcal Y.
$$

记号 $h_S$ 强调学得的规则依赖于随机样本 $S$。

对于损失函数（loss function）$\ell$，总体风险（population risk）、真实风险（true risk）或泛化误差（generalization error）定义为

$$
L_{\mathbb P}(h)
=\mathbb E_{(X,Y)\sim\mathbb P}[\ell(h,X,Y)].
$$

经验风险（empirical risk）为

$$
L_S(h)=\frac1n\sum_{i=1}^n\ell(h,X_i,Y_i).
$$

对于二分类（binary classification），$0$--$1$ 损失（zero-one loss）为

$$
\ell_{0/1}(h,(x,y))
=\mathbf 1\{h(x)\ne y\}
=
\begin{cases}
1,&h(x)\ne y,\\
0,&h(x)=y.
\end{cases}
$$

因此

$$
L_{\mathbb P}(h)=\mathbb P(h(X)\ne Y).
$$

总体风险是我们真正关心的量，但由于 $\mathbb P$ 未知，通常无法计算。经验风险可从数据中观察，所以被用作代理量（proxy）。

## 2.1 补充：贝叶斯分类器

当 $\mathcal Y=\{0,1\}$ 时，定义

$$
\eta(x)=\mathbb P(Y=1\mid X=x).
$$

在 $0$--$1$ 损失下，预测 $1$ 的条件错误（conditional error）是 $1-\eta(x)$，预测 $0$ 的条件错误是 $\eta(x)$。因此，贝叶斯分类器（Bayes classifier）为

$$
h_{\mathrm{Bayes}}(x)=\mathbf 1\left\{\eta(x)\ge\frac12\right\}.
$$

它在所有可测分类器（measurable classifier）中使风险最小；但由于数据分布未知，通常无法直接得到它。后面的没有免费午餐定理（No-Free-Lunch theorem, NFL theorem）将解释，为什么任何数据驱动方法都不可能在每一个可能的问题上统一恢复这一理想规则。


# 3. ERM 与核心泛化问题

*幻灯片主线：L7-8，第 7--11 页。*

给定假设类 $\mathcal H$，经验风险最小化返回

$$
h_S\in\operatorname*{argmin}_{h\in\mathcal H}L_S(h).
$$

核心问题是：

> 如果 ERM 使 $L_S(h_S)$ 很小，那么这对 $L_{\mathbb P}(h_S)$ 意味着什么？

希望得到的保证是

$$
L_{\mathbb P}(h_S)
\le
\inf_{h\in\mathcal H}L_{\mathbb P}(h)+\varepsilon.
$$

如果 $L_S(h)$ 对 $\mathcal H$ 中的**每一个** $h$ 都同时近似 $L_{\mathbb P}(h)$，ERM 就无法利用经验风险偶然向下波动（downward fluctuation）的机会。这一思想稍后将成为一致收敛。

对于一个在观察 $S$ 之前就固定的假设 $h$，

$$
\mathbb E_S[L_S(h)]=L_{\mathbb P}(h).
$$

这种无偏性（unbiasedness）对于 $h_S$ 并不足够，因为 $h_S$ 是在看到数据以后选出的，而同一批数据又用于计算 $L_S$。学习理论必须控制这种数据依赖性（data dependence）。幻灯片也提醒我们，假设类层面的分析只是一个视角：优化过程以及算法最终选择的具体解同样可能重要。


# 4. 归纳偏置：限制 ERM 的搜索范围

*幻灯片主线：L7-8，第 8 页。*

若 ERM 在所有函数上搜索，它可以记忆样本，并在样本之外任意行动。归纳偏置要求在观察训练数据之前就限制搜索空间。例子包括线性模型（linear model）、神经网络架构（neural-network architecture）与随机森林（random forest）。

此时学习规则为

$$
\operatorname{ERM}_{\mathcal H}(S)
\in\operatorname*{argmin}_{h\in\mathcal H}L_S(h).
$$

理想情况下，$\mathcal H$ 应反映关于数据的先验知识（prior knowledge）。假设类过于丰富时仍可能过拟合，过度受限时则可能欠拟合（underfitting）。不存在没有任何归纳偏置的学习；后面的 NFL 定理将把这一必要性形式化。


# 5. 近似误差与估计误差

*幻灯片主线：L7-8，第 12--17 页。*

令

$$
h_{\mathcal H}^*
\in\operatorname*{argmin}_{h\in\mathcal H}L_{\mathbb P}(h).
$$

则

$$
\begin{aligned}
L_{\mathbb P}(h_S)
&=L_{\mathbb P}(h_{\mathcal H}^*)
+\left[L_{\mathbb P}(h_S)-L_{\mathbb P}(h_{\mathcal H}^*)\right]\\
&=\varepsilon_{\mathrm{apx}}+\varepsilon_{\mathrm{est}},
\end{aligned}
$$

其中

$$
\varepsilon_{\mathrm{apx}}
=\inf_{h\in\mathcal H}L_{\mathbb P}(h)
$$

是近似误差（approximation error），而

$$
\varepsilon_{\mathrm{est}}
=L_{\mathbb P}(h_S)-\inf_{h\in\mathcal H}L_{\mathbb P}(h)
$$

是估计误差（estimation error）。

近似误差衡量归纳偏置所付出的代价：它取决于 $\mathcal H$ 表达底层关系的能力，并不直接取决于样本量 $n$ 或这一次实际抽到的样本。估计误差衡量从有限数据学习所付出的代价：经验风险只是总体风险的代理。

幻灯片要求读者考察：

- 先验知识与 $\mathcal H$ 的选择如何影响近似误差；
- 假设类容量（class capacity）、样本量（sample size）、实际样本、分布偏移（distribution shift）与鲁棒性（robustness）如何影响估计误差。

**补充。** 若 ERM 只能近似求解，还可以加入优化误差（optimization error）。对于算法输出 $\widetilde h_S$，

$$
L_{\mathbb P}(\widetilde h_S)-L_{\mathbb P}(h_{\mathrm{Bayes}})
=\underbrace{L_{\mathbb P}(h_{\mathcal H}^*)-L_{\mathbb P}(h_{\mathrm{Bayes}})}_{\text{近似误差}}
+\underbrace{L_{\mathbb P}(h_S)-L_{\mathbb P}(h_{\mathcal H}^*)}_{\text{估计误差}}
+\underbrace{L_{\mathbb P}(\widetilde h_S)-L_{\mathbb P}(h_S)}_{\text{优化误差}}.
$$


# 6. 经典容量权衡与双降现象

*幻灯片主线：L7-8，第 18--19 页。*

经典图像的横轴是模型容量（model capacity），纵轴是风险。容量增大时，训练风险（training risk）下降；测试风险（test risk）呈 U 形：弱假设类会欠拟合，中等容量给出“最佳点”（sweet spot），非常丰富的假设类则会过拟合。

现代的双降（double descent）图像加入了插值阈值（interpolation threshold）。在参数不足（under-parameterized）的经典区域中，先出现第一条 U 形曲线。当模型刚好首次能把训练风险降到零时，测试风险可能陡增；进入参数过剩（over-parameterized）的现代插值区域（interpolating regime）以后，即使训练风险仍保持为零，测试风险仍可能再次下降。

幻灯片引用 Mikhail Belkin、Daniel Hsu、Siyuan Ma 与 Soumik Mandal 的论文 *Reconciling modern machine-learning practice and the classical bias-variance trade-off*。结论不是误差分解失效，而是原始参数数量可能并非合适的复杂度度量（complexity measure），且优化算法可能通过隐式偏置（implicit bias）选出某些特定的插值解（interpolating solution）。


# 7. 形式化的可学习模型：可实现性与有限假设类

*幻灯片主线：L7-8，第 20--23 页。*

## 7.1 假设一：可实现性

可实现性假设（realizability assumption）是

$$
\exists h^*\in\mathcal H
\quad\text{使得}\quad
L_{\mathbb P}(h^*)=0.
$$

于是 $L_S(h^*)=0$ 几乎处处成立（almost surely, a.s.）。一种证明使用

$$
\mathbb E_S[L_S(h^*)]=L_{\mathbb P}(h^*)=0
$$

以及“非负随机变量（nonnegative random variable）的期望为零，则该变量几乎处处为零”这一事实。由于 $h^*$ 的经验风险为零，每个 ERM 解也都满足

$$
L_S(h_S)=0.
$$

这**并不**意味着 $h_S=h^*$ 或 $L_{\mathbb P}(h_S)=0$：可能有多个假设都能对训练数据插值（interpolate）。

## 7.2 假设二：有限假设类

假设

$$
|\mathcal H|<\infty.
$$

幻灯片把它称为一种很强的归纳偏置。问题是：可实现性与有限性是否足以保证 ERM 能够学习？答案是肯定的。


# 8. 有限可实现假设类定理

*幻灯片主线：L7-8，第 24、29 页。*

若可实现性成立且 $|\mathcal H|<\infty$，则每一个 ERM 解都满足

$$
\mathbb P_{S\sim\mathbb P^n}
\left(L_{\mathbb P}(h_S)>\varepsilon\right)
\le |\mathcal H|e^{-n\varepsilon}.
$$

因此，只要

$$
n\ge\frac{\log(|\mathcal H|/\delta)}{\varepsilon},
$$

就以至少 $1-\delta$ 的概率有

$$
L_{\mathbb P}(h_S)\le\varepsilon.
$$

两个参数的作用不同：$\varepsilon$ 是精度容差（accuracy tolerance），$\delta$ 是失败概率（failure probability），亦即置信参数（confidence parameter）。该定理是统计性结论，并没有说明 ERM 能否被高效计算。


# 9. 利用误导性样本与并集界的完整证明

*幻灯片主线：L7-8，第 25--28 页。*

定义坏假设（bad hypothesis）的集合

$$
\mathcal H_B
=\{h\in\mathcal H:L_{\mathbb P}(h)>\varepsilon\}
$$

以及误导性样本事件（misleading-sample event）

$$
M
=\{S:\exists h\in\mathcal H_B\text{ with }L_S(h)=0\}.
$$

可实现性给出 $L_S(h_S)=0$。因此，只要 ERM 失败，$h_S$ 本身就是一个在样本上看似完美的坏假设：

$$
\{S:L_{\mathbb P}(h_S)>\varepsilon\}\subseteq M.
$$

对于固定的 $h\in\mathcal H_B$，

$$
\begin{aligned}
\mathbb P^n(L_S(h)=0)
&=(1-L_{\mathbb P}(h))^n\\
&\le(1-\varepsilon)^n\\
&\le e^{-n\varepsilon}.
\end{aligned}
$$

独立性（independence）带来概率乘积，$1-u\le e^{-u}$ 给出最后一个不等式。由于

$$
M=\bigcup_{h\in\mathcal H_B}\{S:L_S(h)=0\},
$$

并集界（union bound）给出

$$
\begin{aligned}
\mathbb P^n(L_{\mathbb P}(h_S)>\varepsilon)
&\le\mathbb P^n(M)\\
&\le\sum_{h\in\mathcal H_B}\mathbb P^n(L_S(h)=0)\\
&\le|\mathcal H_B|e^{-n\varepsilon}\\
&\le|\mathcal H|e^{-n\varepsilon}.
\end{aligned}
$$

要求最后一个量不超过 $\delta$，就得到定理。第 25 页的抛硬币直觉是：许多真正很差的候选者同时、独立地走运，概率会随样本量指数衰减（exponential decay）。

L7-8 第 25 页的证明概要引用 Shalev-Shwartz 与 Ben-David 教材第 2.3.1 节；幻灯片把该教材缩写为 **[SSS]**。


# 10. PAC 学习

*幻灯片主线：L7-8，第 29--31 页；L9，第 2--3 页重复出现。*

在可实现性假设下，如果存在一个学习算法和一个样本复杂度函数（sample-complexity function）$m_{\mathcal H}(\varepsilon,\delta)$，使得对于每一组 $\varepsilon,\delta\in(0,1)$ 和每一个可实现分布 $\mathbb P$，只要

$$
n\ge m_{\mathcal H}(\varepsilon,\delta),
$$

算法输出 $h$ 就满足

$$
\mathbb P_{S\sim\mathbb P^n}
\left(L_{\mathbb P}(h)\le\varepsilon\right)
\ge1-\delta,
$$

则称 $\mathcal H$ 是 PAC 可学习的（PAC learnable）。

之所以只能说“很可能”（probably），是因为随机的有限样本可能不具代表性；之所以只能说“近似正确”（approximately correct），是因为有限样本可能遗漏相关细节。幻灯片展示了提出 PAC 框架（PAC framework）的 Leslie Valiant，并注明他获得了 2010 年图灵奖（Turing Award）。

还剩两个问题：如果可实现性不成立怎么办？如果 $\mathcal H$ 是无限的又怎么办？


# 11. 没有免费午餐：为什么归纳偏置不可避免

*幻灯片主线：L7-8，第 33 页；L9，第 4 页重复出现。*

幻灯片把下面的结果标为 **定理 5.1（没有免费午餐，Theorem 5.1: No-Free-Lunch）**。对于 $0$--$1$ 损失下的任意二分类算法 $A$，如果

$$
m<\frac{|\mathcal X|}{2},
$$

那么存在 $\mathcal X\times\{0,1\}$ 上的分布 $\mathcal D$，使得

1. 某个函数 $f:\mathcal X\to\{0,1\}$ 满足 $L_{\mathcal D}(f)=0$；
2. 对 $S\sim\mathcal D^m$，以至少 $1/7$ 的概率有

   $$
   L_{\mathcal D}(A(S))\ge\frac18.
   $$

这些常数并非概念重点。定理说的是：对于每一个学习器（learner），总存在一个使它失败的任务，尽管一个带有不同任务特定偏置（task-specific bias）的学习器可能成功。不存在一个在所有可能的数据生成机制（data-generating mechanism）上都占优的通用学习器。

**幻灯片练习。** 给出一个在构造出的任务上成功、而 $A$ 失败的学习器。

**答案。** 定理保证存在一个零风险目标 $f$。把 $f$ 硬编码（hard-code）进规则 $B(S)=f$，就得到零风险。这是存在性论证（existence argument），并不是说实际学习器事先知道 $f$；它揭示了先验信息的作用。


# 12. 不可知 PAC 学习（agnostic PAC learning）

*幻灯片主线：L7-8，第 34--35 页；L9，第 5 页。*

不可知 PAC 学习不再假设目标规律能够由 $\mathcal H$ 完全实现。

在没有可实现性的情形下，绝对风险可能根本无法小于 $\varepsilon$。更弱且有意义的目标，是与所选假设类内的最优假设比较：

$$
L_{\mathbb P}(h_S)
\le
\inf_{h\in\mathcal H}L_{\mathbb P}(h)+\varepsilon.
$$

如果存在一个算法，使得对每一个分布 $\mathbb P$，只要 $n\ge m_{\mathcal H}(\varepsilon,\delta)$，上述不等式就以至少 $1-\delta$ 的概率成立，则称该类是不可知 PAC 可学习的（agnostic PAC learnable）。这一定义适用于一般风险

$$
L_{\mathbb P}(h)=\mathbb E_{\mathbb P}[\ell(h,X,Y)],
$$

而不仅限于 $0$--$1$ 分类。

在所选假设类内，近似误差现在是不可约的（irreducible）。学习所控制的是对所有分布都成立的估计误差。这种无分布假设（distribution-free）的强度同时也是局限：由于不能利用实际分布的良性性质（benign property），界可能较为保守。L7-8 第 35 页进一步指向 [SSS] 第 6.1 节，其中讨论了无限但仍可学习的假设类。


# 13. 代表性样本与一致收敛

*幻灯片主线：L7-8，第 36--38 页；L9，第 6--8 页。*

如果样本 $S$ 满足

$$
\sup_{h\in\mathcal H}
|L_S(h)-L_{\mathbb P}(h)|
\le\eta,
$$

则称它是关于 $\mathcal H$ 的 $\eta$-代表性样本（$\eta$-representative sample）。

如果 $S$ 是 $\varepsilon/2$-代表性的，且 $h_S$ 是 ERM 解，那么对于每一个 $h\in\mathcal H$，

$$
\begin{aligned}
L_{\mathbb P}(h_S)
&\le L_S(h_S)+\frac\varepsilon2\\
&\le L_S(h)+\frac\varepsilon2\\
&\le L_{\mathbb P}(h)+\varepsilon.
\end{aligned}
$$

对 $h$ 取下确界（infimum）可得

$$
L_{\mathbb P}(h_S)
\le
\inf_{h\in\mathcal H}L_{\mathbb P}(h)+\varepsilon.
$$

因此，一致收敛足以保证 ERM 是不可知 PAC 学习器。幻灯片把正式陈述标为 **定义 4.3（一致收敛，Definition 4.3: Uniform Convergence）**：如果存在 $m_{\mathcal H}^{\mathrm{UC}}(\eta,\delta)$，使得对于每一个 $\mathbb P$，只要 $n\ge m_{\mathcal H}^{\mathrm{UC}}(\eta,\delta)$，便有

$$
\mathbb P_{S\sim\mathbb P^n}
\left(
\sup_{h\in\mathcal H}|L_S(h)-L_{\mathbb P}(h)|\le\eta
\right)
\ge1-\delta,
$$

则称 $\mathcal H$ 具有一致收敛性质（uniform convergence property）。

幻灯片文字有时把样本称作 $\varepsilon$-代表性的，而引理使用 $\varepsilon/2$。精确规则是：一致偏差（uniform deviation）不超过 $\eta$，会使 ERM 的超额风险（excess risk）不超过 $2\eta$。

## 13.1 补充：有限假设类的一致收敛

对于取值于 $[0,1]$ 的有界损失（bounded loss），Hoeffding 不等式（Hoeffding's inequality）对固定 $h$ 给出

$$
\mathbb P\left(|L_S(h)-L_{\mathbb P}(h)|>\eta\right)
\le2e^{-2n\eta^2}.
$$

对有限 $\mathcal H$ 使用并集界可得

$$
\mathbb P\left(
\sup_{h\in\mathcal H}|L_S(h)-L_{\mathbb P}(h)|>\eta
\right)
\le2|\mathcal H|e^{-2n\eta^2}.
$$

这已经给出了不可知情形的尺度

$$
n=O\left(\frac{\log|\mathcal H|+\log(1/\delta)}{\varepsilon^2}\right).
$$

下一讲将用有限数据点上不同标记方式的有效数量，替代假设类的字面有限基数（cardinality）。


# 第二部分——L9：增长函数、打散（shattering）与 VC 维


# 14. L9 对 PAC 学习的两项推广

*幻灯片主线：L9，第 1--9 页。*

L9 首先重复 PAC 定义以及 $\varepsilon$、$\delta$ 各自的作用，然后展开两项推广：

1. 用不可知情形下的比较

   $$
   L_{\mathbb P}(h_S)
   \le
   \inf_{h\in\mathcal H}L_{\mathbb P}(h)+\varepsilon
   $$

   代替可实现性；

2. 用假设类在有限个点上能够表现出的不同行为数量，代替对无限假设类毫无用处的字面大小 $|\mathcal H|$。

L9 第 4--8 页的 NFL 定理、代表性样本引理与一致收敛的正式定义，正是第 11--13 节建立的桥梁。L9 第 9 页宣布，合适的复杂度概念将是 VC 维。


# 15. 有效大小与增长函数

*幻灯片主线：L9，第 10--12 页。*

L9 第 10--16 页注明，这组增长函数幻灯片来自 **Y. Polyanskiy（MIT）**。

考虑齐次超平面分类器（homogeneous hyperplane classifier）

$$
\widehat Y(x)=\operatorname{sgn}(x^T w).
$$

参数向量（parameter vector）$w$ 有无穷多个。然而，一旦固定 $x^{(1)},\ldots,x^{(N)}$，能够出现的标签向量（label vector）只有有限多个。这个数量，而不是参数空间（parameter space）的基数，才是假设类在数据上的有效大小（effective size）。

对于 $C=\{c_1,\ldots,c_m\}\subseteq\mathcal X$，定义限制（restriction）

$$
\mathcal H_C
=\{(h(c_1),\ldots,h(c_m)):h\in\mathcal H\}.
$$

增长函数在 L10 第 5 页再次被标为 **定义 6.9（Definition 6.9）**，其定义为

$$
\tau_{\mathcal H}(m)
=\max_{\substack{C\subseteq\mathcal X\\|C|=m}}|\mathcal H_C|.
$$

等价地，

$$
\tau_{\mathcal H}(m)
=\max_{x_1,\ldots,x_m}
\left|
\{(h(x_1),\ldots,h(x_m)):h\in\mathcal H\}
\right|.
$$

总有

$$
1\le\tau_{\mathcal H}(m)\le2^m.
$$

幻灯片的关键主张是：有限类论证中的 $|\mathcal H|$，往往可以换成 $\tau_{\mathcal H}(N)$。第 12 页还有一句孤立的“勒贝格积分（Lebesgue integral）”旁注；这是教师的随堂插话，并不是增长函数定义的一部分。


# 16. 一维仿射分类器

*幻灯片主线：L9，第 13 页；L10，第 6 页。*

对于阈值方向可以反转的一维仿射分类器（one-dimensional affine classifier），两个点可以取得全部四种二元标记（binary labeling）：

$$
\tau_{\mathcal H}(2)=2^2=4.
$$

对于三个有序点，交替模式 $(+,-,+)$ 与 $(-,+,-)$ 无法用单一决策边界（decision boundary）实现，其余六种模式都可以实现，因此

$$
\tau_{\mathcal H}(3)=6<2^3.
$$

更一般地，这个双向仿射类在 $m\ge1$ 时满足 $\tau_{\mathcal H}(m)=2m$。不要把它与单向阈值类（one-direction threshold class）$h_a(x)=\mathbf1\{x<a\}$ 混淆；后者只有 $m+1$ 种模式，VC 维为 $1$。


# 17. 增长函数的二分现象

*幻灯片主线：L9，第 14--17 页。*

Vapnik--Chervonenkis 二分现象（Vapnik--Chervonenkis dichotomy）指出，一个二分类假设类只有两种行为之一：

- 对每个 $m$ 都有 $\tau_{\mathcal H}(m)=2^m$；或
- 从某个有限位置开始，$\tau_{\mathcal H}(m)$ 至多以多项式速度增长（polynomial growth）。

当多项式指数为 $d$ 时，幻灯片给出了可实现情形下的启发式估计（heuristic）

$$
\varepsilon
<\frac1N\log\frac{\tau_{\mathcal H}(N)}{\delta}
=O\left(
\frac dN\log\frac Nd+
\frac1N\log\frac1\delta
\right).
$$

这是尺度估计（scale calculation），而不是最紧的不可知学习结果。标准的可实现情形上界为

$$
N=O\left(
\frac{d\log(1/\varepsilon)+\log(1/\delta)}{\varepsilon}
\right),
$$

而不可知估计则具有 $1/\varepsilon^2$ 的尺度。

如果对每个 $m$ 都有 $\tau_{\mathcal H}(m)=2^m$，该类可以在任意有限集合上赋予任意标签。在一种最坏情形构造（worst-case construction）中，样本没有提供关于下一个未见点的任何信息，于是该点上的错误率为 $1/2$，与随机猜测（random guess）相同。这是 NFL 的直觉，并不是说每个算法在每个固定分布下的风险都必定恰好为 $1/2$。

幻灯片的总结十分关键：可学习性取决于假设类在有限子集上最多能实现多少种分类，而不是其字面基数。能力过强时，可以构造一个坏分布，使观察到的点仍保持可实现性，却令学习器在其他地方失败。


# 18. 打散与 VC 维

*幻灯片主线：L9，第 18 页。*

如果有限集合 $C=\{c_1,\ldots,c_m\}$ 的每一种二元标记都能由 $\mathcal H$ 实现，则称 $\mathcal H$ 打散了 $C$：

$$
\forall(y_1,\ldots,y_m)\in\{0,1\}^m,
\quad
\exists h\in\mathcal H,
\quad
h(c_i)=y_i\ \text{对所有 }i.
$$

等价地，

$$
|\mathcal H_C|=2^{|C|}.
$$

Vapnik--Chervonenkis 维定义为

$$
\operatorname{VCdim}(\mathcal H)
=\sup\{|C|:\mathcal H\text{ shatters }C\}.
$$

量词（quantifier）非常重要。要证明下界，只需找到一个能被打散的合适集合；要证明上界，则必须证明**每一个**更大的集合都不能被打散。无限 VC 维（infinite VC dimension）意味着 $\mathcal H$ 不是 PAC 可学习的。


# 19. 打散的例子与参数的作用

*幻灯片主线：L9，第 19 页。*

$\mathbb R^2$ 中三个不共线点可被仿射线性分类器（affine linear classifier）打散。直线可以隔离任意一个点，反转直线的方向则会交换正、负标签。幻灯片用有向直线（oriented line）展示了这些分割；图像署名为 C. Burges。

任何四个点都不能被打散。如果四点处于凸位置（convex position），交替的异或标记（exclusive-or labeling, XOR labeling）不是线性可分的（linearly separable）。如果一个点位于其余三个点的凸包（convex hull）内，那么把内部点标成与三个外部点相反的类别也不可能实现。因此

$$
\operatorname{VCdim}(\mathbb R^2\text{ 中的仿射半空间})=3.
$$

**补充：一个参数仍可能具有无限 VC 维。** 参数数量不是普适的容量度量。一个经典振荡函数族（oscillatory family）形如

$$
h_\alpha(x)=\operatorname{sign}(\sin(\alpha x)).
$$

通过精心选择输入，只改变一个频率参数（frequency parameter）$\alpha$，也可以实现任意多种标记。这个例子来自此前笔记所依据的教材内容，在渲染后的 L9 第 19 页上并不可见。


# 20. 如何证明一个 VC 维结论

*幻灯片主线：L9，第 20 页。*

要证明 $\operatorname{VCdim}(\mathcal H)=d$，必须同时证明：

1. 存在一个大小为 $d$ 且被 $\mathcal H$ 打散的集合；
2. 大小为 $d+1$ 的任何集合都不能被 $\mathcal H$ 打散。

因此，要证明 $d_1\le\operatorname{VCdim}(\mathcal H)\le d_2$，需要给出一个可打散的 $d_1$ 点见证集合（witness set），并对每一个 $(d_2+1)$ 点集合给出不可能性证明。只证明某一个特定的大集合不能被打散，不能建立上界。


# 21. 阈值函数与有限假设类

*幻灯片主线：L9，第 21 页。*

幻灯片把该阈值假设类标为 **例 6.2（Example 6.2）**。考虑

$$
\mathcal H=\{h_a:a\in\mathbb R\},
\qquad
h_a(x)=\mathbf1\{x<a\}.
$$

一个点可被打散：把阈值放在该点任意一侧即可。如果 $c_1\le c_2$，标记 $(0,1)$ 不可能出现，因为 $h_a(c_1)=0$ 意味着 $a\le c_1\le c_2$，从而 $h_a(c_2)=0$。因此

$$
\operatorname{VCdim}(\mathcal H)=1.
$$

这个假设类虽然无限，却是 PAC 可学习的。

对于有限假设类，若 $d$ 个点被打散，那么在这些点上至少需要 $2^d$ 个不同假设。因此

$$
2^d\le|\mathcal H|
\quad\Longrightarrow\quad
\operatorname{VCdim}(\mathcal H)\le\log_2|\mathcal H|.
$$

幻灯片明确要求学生阅读 Shalev-Shwartz 与 Ben-David 教材的第 6 章及其练习。


# 22. 统计学习基本定理

*幻灯片主线：L9，第 22 页；L10，第 2 页重复并给出定量版本。*

幻灯片把该结果标为 **定理 6.7（统计学习基本定理，Theorem 6.7: The Fundamental Theorem of Statistical Learning）**。对于使用 $0$--$1$ 损失的二分类，以下六个条件等价：

1. $\mathcal H$ 具有一致收敛性质。
2. $\mathcal H$ 的每一个 ERM 规则都是成功的不可知 PAC 学习器。
3. $\mathcal H$ 是不可知 PAC 可学习的。
4. $\mathcal H$ 是 PAC 可学习的。
5. $\mathcal H$ 的每一个 ERM 规则都是成功的 PAC 学习器。
6. $\operatorname{VCdim}(\mathcal H)<\infty$。

幻灯片标出的简单蕴含关系（implication）是

$$
1\Rightarrow2\Rightarrow3\Rightarrow4,
\qquad
2\Rightarrow5.
$$

$4\Rightarrow6$ 与 $5\Rightarrow6$ 可由 NFL 构造推出。L10 证明困难方向 $6\Rightarrow1$。

## 22.1 定量版本

**定理 6.8（定量版本，Theorem 6.8: Quantitative Version），展示于 L10 第 2 页。** 如果 $d=\operatorname{VCdim}(\mathcal H)<\infty$，那么存在绝对常数（absolute constant）$C_1,C_2$，使得

$$
m_{\mathcal H}^{\mathrm{UC}}(\varepsilon,\delta)
=\Theta\left(\frac{d+\log(1/\delta)}{\varepsilon^2}\right),
$$

$$
m_{\mathcal H}^{\mathrm{agnostic}}(\varepsilon,\delta)
=\Theta\left(\frac{d+\log(1/\delta)}{\varepsilon^2}\right),
$$

而在可实现情形下

$$
C_1\frac{d+\log(1/\delta)}{\varepsilon}
\le
m_{\mathcal H}(\varepsilon,\delta)
\le
C_2\frac{d\log(1/\varepsilon)+\log(1/\delta)}{\varepsilon}.
$$

不可知情形需要估计两个非零风险之间的微小差别，所以具有均值估计（mean estimation）的 $1/\varepsilon^2$ 尺度。在可实现情形中，坏假设必须在所有已观察样本上都不犯错；该事件的概率像 $e^{-n\varepsilon}$ 一样衰减，因此产生更快的 $1/\varepsilon$ 尺度。


# 23. 其他复杂度概念与 PAC-Bayes

*幻灯片主线：L9，第 23--24 页。*

L9 最后回顾了代表性样本、打散、VC 维以及证明 VC 维所需的上下界两步模板，随后列出其他复杂度工具：

- Pollard 伪维（Pollard pseudo-dimension），它通过阈值把 VC 维推广到实值函数（real-valued function）；
- 覆盖数（covering number），即在给定分辨率下覆盖函数类所需的度量球（metric ball）数量；
- 度量熵（metric entropy），即覆盖数的对数；
- Rademacher 复杂度（Rademacher complexity），即函数类与随机符号相关的能力；
- Gaussian 复杂度（Gaussian complexity），即把随机符号换成高斯乘子（Gaussian multiplier）所得的类似概念；
- 局部 Rademacher 复杂度（local Rademacher complexity），它度量相关的低风险邻域，而不是整个函数类。

PAC-Bayes 方法（PAC-Bayes method）在假设上设置先验分布（prior distribution），并分析依赖数据的后验分布（posterior distribution）。它的界通常结合经验风险、KL 散度（Kullback--Leibler divergence, KL divergence）复杂度项与置信项。幻灯片强调，这类泛化界可以依赖数据，并指向 Shalev-Shwartz 与 Ben-David 教材第 13 章。


# 第三部分——L10：有限 VC 维、一致收敛与稳定性


# 24. $6\Rightarrow1$ 的证明策略

*幻灯片主线：L10，第 1--3 页。*

L10 证明困难方向

$$
\operatorname{VCdim}(\mathcal H)<\infty
\Longrightarrow
\mathcal H\text{ 具有一致收敛性质}.
$$

关键链条为

$$
\text{有限 VC 维}
\Longrightarrow
\text{多项式增长函数}
\Longrightarrow
\text{趋于零的一致差异（uniform discrepancy）}.
$$

第 2 页展示了非正式速率 $O(\log\tau_{\mathcal H}(N)/\sqrt N)$；正式定理使用的是 $\sqrt{\log\tau_{\mathcal H}(N)}/\sqrt N$。下面写出真正被证明的正式陈述。

**定理 6.11（增长函数一致差异界，Theorem 6.11: growth-function uniform-discrepancy bound）。** 对于每一个分布和 $\delta\in(0,1)$，以至少 $1-\delta$ 的概率有

$$
\sup_{h\in\mathcal H}|L_{\mathbb P}(h)-L_S(h)|
\le
\frac{4+\sqrt{\log\tau_{\mathcal H}(2N)}}{\delta\sqrt{2N}}.
$$

如果 $d=\operatorname{VCdim}(\mathcal H)<\infty$，Sauer 引理（Sauer's lemma；全称 Sauer--Shelah--Perles lemma）给出

$$
\tau_{\mathcal H}(2N)
\le\left(\frac{2eN}{d}\right)^d,
$$

所以该定理蕴含

$$
N_{\mathcal H}^{\mathrm{UC}}(\varepsilon,\delta)
\le\widetilde O\left(\frac{d}{(\delta\varepsilon)^2}\right).
$$

**幻灯片练习：推导这一推论。** Sauer 引理首先给出

$$
\log\tau_{\mathcal H}(2N)
\le
d\log\left(\frac{2eN}{d}\right).
$$

把它代入定理 6.11，可得

$$
Z(S)
\le
\frac{4+\sqrt{d\log(2eN/d)}}{\delta\sqrt{2N}}.
$$

选择 $N$，使右侧不超过 $\varepsilon$。由于 $N$ 同时出现在对数内部，求解该不等式得到软阶上界（soft-order bound）

$$
N=\widetilde O\left(\frac{d}{\delta^2\varepsilon^2}\right).
$$

这里关于 $\delta$ 的依赖较松，是因为证明最终使用了 Markov 不等式（Markov's inequality）。更强的集中不等式（concentration inequality）可以给出第 22.1 节中的最优 $\log(1/\delta)$ 依赖。


# 25. 限制、打散与集合系统表述

*幻灯片主线：L10，第 4--5 页。*

**定义 6.2（限制，Definition 6.2: Restriction）。** 对于 $C=\{c_1,\ldots,c_m\}$，

$$
\mathcal H_C
=\{(h(c_1),\ldots,h(c_m)):h\in\mathcal H\}.
$$

这里的“函数”可以理解为对 $C$ 中各点的一种标记。**定义 6.3（打散，Definition 6.3: Shattering）**指出，$\mathcal H$ 打散 $C$，当且仅当 $|\mathcal H_C|=2^{|C|}$。

它还有一个等价的集合系统表述（set-system formulation）。把每个分类器对应到其正类区域（positive region）

$$
S_h=\{x:h(x)=1\},
\qquad
\mathcal F=\{S_h:h\in\mathcal H\}.
$$

那么 $\mathcal H$ 打散 $C$，当且仅当

$$
\{C\cap S:S\in\mathcal F\}=2^C.
$$

原因是，一种二元标记可由 $C$ 中被赋予标签 $1$ 的子集唯一确定。这也回答了第 4 页关于两个打散定义等价性的练习。幻灯片还非正式地使用“覆盖（covering）”一词表示实现所有子集；这里不要把它与度量覆盖数混淆。


# 26. Sauer--Shelah--Perles 引理

*幻灯片主线：L10，第 5--7 页。*

**引理 6.10（Sauer--Shelah--Perles）。** 如果 $\operatorname{VCdim}(\mathcal H)\le d<\infty$，那么

$$
\tau_{\mathcal H}(m)
\le
\sum_{i=0}^{d}\binom mi.
$$

当 $1\le d<m$ 时，

$$
\sum_{i=0}^{d}\binom mi
\le
\left(\frac{em}{d}\right)^d.
$$

因此，有限 VC 维迫使增长函数呈多项式增长，而不是指数增长（exponential growth）。无限 VC 维则给出每个 $m$ 上的 $\tau_{\mathcal H}(m)=2^m$。

**幻灯片练习：证明第二个不等式。** 下面是一种标准证明。当 $d\le m/2$ 时，取 $x=d/(m-d)\le1$。由于对 $i\le d$ 有 $x^i\ge x^d$，

$$
x^d\sum_{i=0}^d\binom mi
\le
\sum_{i=0}^d\binom mi x^i
\le(1+x)^m.
$$

因此


$$
\sum_{i=0}^d\binom mi
\le
\left(\frac md\right)^d
\left(\frac m{m-d}\right)^{m-d}
\le
\left(\frac{em}{d}\right)^d,
$$

其中 $(1+d/(m-d))^{m-d}\le e^d$。若 $d>m/2$，则使用 $\sum_{i=0}^d\binom mi\le2^m$ 以及

$$
2^m\le\left(\frac{em}{d}\right)^d;
$$

后一个不等式可通过令 $r=d/m\in(1/2,1)$，再观察 $r(1-\log r)\ge\log2$ 得到。

例子

$$
\mathcal F=\{S\subseteq[m]:|S|\le d\}
$$

的大小恰好为 $\sum_{i=0}^d\binom mi$，且不能打散任何 $(d+1)$ 点集合，因为它没有包含完整的 $(d+1)$ 点子集。这说明组合上界（combinatorial bound）是紧的（tight）。

## 26.1 补充：递归证明

令 $g_d(m)$ 表示 VC 维至多为 $d$ 的假设类在 $m$ 个点上最多能产生的标记数。把前 $m-1$ 个点上的标记分为两类：对最后一个点只有一种扩展的标记，以及对最后一个点有两种扩展的标记。后一类的 VC 维至多为 $d-1$；否则再加入最后一个点便会打散 $d+1$ 个点。因此

$$
g_d(m)\le g_d(m-1)+g_{d-1}(m-1).
$$

结合 $g_0(m)=g_d(0)=1$ 与 Pascal 恒等式（Pascal's identity），可得

$$
g_d(m)\le\sum_{i=0}^d\binom mi.
$$

幻灯片中的历史注释是：

- 1968 年：Vapnik 与 Chervonenkis 发表结果，证明发表于 1969 年；
- 1970 年：Erdős 在国际数学家大会（International Congress of Mathematicians, ICM）上提出这个问题；
- 1972 年：Sauer 在 *Journal of Combinatorial Theory (A)* 上回答了该问题；
- 幽默评论：“Nobody can find it in Shelah.”（没有人能在 Shelah 的论文中找到它。）


# 27. 从期望差异到高概率结论

*幻灯片主线：L10，第 8--9 页。*

定义

$$
Z(S)=\sup_{h\in\mathcal H}|L_{\mathbb P}(h)-L_S(h)|.
$$

证明首先建立期望界（expectation bound）

$$
\mathbb E_S[Z(S)]
\le
\frac{4+\sqrt{\log\tau_{\mathcal H}(2m)}}{\sqrt{2m}}.
$$

由于 $Z\ge0$，Markov 不等式给出

$$
\mathbb P\left(Z>\frac{\mathbb E Z}{\delta}\right)\le\delta,
$$

从而得到第 3 页的定理。幻灯片把该结果和证明计划归于 [SSS] 的**定理 6.11**。

第 9 页列出的六个证明步骤是：

1. 用幽灵样本进行变量加倍（variable doubling with a ghost sample）；
2. Jensen 不等式（Jensen's inequality）；
3. 用随机符号进行对称化（symmetrization）；
4. 限制到两个样本中出现的有限集合；
5. Hoeffding 不等式加并集界；
6. 把概率尾界（probability tail bound）转换为期望界。


# 28. 第一步：用幽灵样本进行变量加倍

*幻灯片主线：L10，第 10--11 页。*

令幽灵样本（ghost sample）$S'=(Z_1',\ldots,Z_m')\sim\mathbb P^m$ 与 $S$ 相互独立。对于固定的 $h$，

$$
L_{\mathbb P}(h)=\mathbb E_{S'}[L_{S'}(h)].
$$

因此

$$
\mathbb E_S\sup_h|L_{\mathbb P}(h)-L_S(h)|
=
\mathbb E_S\sup_h
\left|
\mathbb E_{S'}[L_{S'}(h)-L_S(h)]
\right|.
$$

未知的总体期望（population expectation）由另一个经验样本上的期望代替。


# 29. 第二步：Jensen 不等式与经验风险展开

*幻灯片主线：L10，第 12--14 页。*

利用 $|\mathbb EX|\le\mathbb E|X|$ 以及 $\sup_h\mathbb E f_h\le\mathbb E\sup_h f_h$，

$$
\mathbb E_S\sup_h|L_{\mathbb P}(h)-L_S(h)|
\le
\mathbb E_{S,S'}\sup_h|L_{S'}(h)-L_S(h)|.
$$

写成 $z_i=(x_i,y_i)$、$z_i'=(x_i',y_i')$ 后得到

$$
\mathbb E_{S,S'}
\left[
\sup_{h\in\mathcal H}
\left|
\frac1m\sum_{i=1}^m
\bigl(\ell(h,z_i')-\ell(h,z_i)\bigr)
\right|
\right].
$$

第 13、14 页是配合证明节奏的页面：前者重述上面的界，后者代入经验风险的定义。


# 30. 第三步：对称化技巧

*幻灯片主线：L10，第 15--16 页。*

令 $\sigma_1,\ldots,\sigma_m$ 为相互独立的 Rademacher 随机符号（Rademacher sign），每个都在 $\{-1,+1\}$ 上均匀分布。由于每一对 $(z_i,z_i')$ 都是可交换的（exchangeable），交换这一对会使

$$
\ell(h,z_i')-\ell(h,z_i)
$$

改变符号，却不改变其分布。因此上一步的期望等于

$$
\mathbb E_{S,S',\sigma}
\left[
\sup_{h\in\mathcal H}
\left|
\frac1m\sum_{i=1}^m
\sigma_i\bigl(\ell(h,z_i')-\ell(h,z_i)\bigr)
\right|
\right].
$$

幻灯片中的骰子图形表示这些随机符号。这正是 Rademacher 复杂度论证的核心机制。


# 31. 第四步：把无限假设类约化为有限限制

*幻灯片主线：L10，第 17 页。*

给定 $S,S'$，令**输入点**（input point）集合为

$$
C_X=\{x_1,\ldots,x_m,x_1',\ldots,x_m'\},
\qquad |C_X|\le2m.
$$

幻灯片中“appearing in both samples”是指两个样本的并集（union），而不是交集（intersection）。只有在 $C_X$ 上的预测会影响当前表达式：在 $C_X$ 上限制相同的两个假设，会在固定的带标签样本上产生相同损失。令 $\widetilde{\mathcal H}_{C_X}\subseteq\mathcal H$ 从每一个不同的限制等价类（restriction class）中选取一个代表，则

$$
|\widetilde{\mathcal H}_{C_X}|
=|\mathcal H_{C_X}|
\le\tau_{\mathcal H}(2m),
$$

且对 $\mathcal H$ 的上确界可换成对这些代表的最大值。这个区别很重要：$\mathcal H_{C_X}$ 本身是标记向量的集合，而所选代表仍然是可以代入 $\ell$ 的函数。

对于 $h\in\widetilde{\mathcal H}_{C_X}$，定义

$$
\theta_h
=\frac1m\sum_{i=1}^m
\sigma_i\bigl(\ell(h,z_i')-\ell(h,z_i)\bigr).
$$

在给定 $S,S'$ 的条件下，$\mathbb E_\sigma[\theta_h]=0$，且各求和项相互独立并有界。


# 32. 第五步：Hoeffding 不等式与并集界

*幻灯片主线：L10，第 18 页。*

**幻灯片练习：证明 Hoeffding 步骤与并集界步骤。** 令

$$
a_i=\ell(h,z_i')-\ell(h,z_i),
\qquad |a_i|\le1.
$$

在给定 $S,S'$ 的条件下，$\theta_h=m^{-1}\sum_i\sigma_i a_i$ 是均值为零的 Rademacher 和（Rademacher sum）。因此，Hoeffding 不等式的一般形式为

$$
\mathbb P_\sigma(|\theta_h|>\rho)
\le2e^{-c m\rho^2}
$$

其中 $c>0$ 是绝对常数。按照标准值域计算（range calculation），$\sigma_i a_i/m\in[-|a_i|/m,|a_i|/m]$，由此得到 $c=1/2$。幻灯片采用不同的值域归一化，并明确写出：对于每个 $\rho>0$，

$$
\mathbb P_\sigma(|\theta_h|>\rho)
\le2e^{-2m\rho^2}.
$$

精确常数不影响学习速率（learning rate）；这里保留 $c=2$，因为它是幻灯片展示的公式。最大值超界的坏事件是各个假设超界事件的并集，因此

$$
\begin{aligned}
&\mathbb P_\sigma\left(
\max_{h\in\widetilde{\mathcal H}_{C_X}}|\theta_h|>\rho
\right)\\
&\quad\le
\sum_{h\in\widetilde{\mathcal H}_{C_X}}
\mathbb P_\sigma(|\theta_h|>\rho)\\
&\quad\le
2|\widetilde{\mathcal H}_{C_X}|e^{-2m\rho^2}.
\end{aligned}
$$

采用幻灯片的简记 $|\mathcal H_C|$ 表示不同限制的数量，则用严谨的代表假设类记号写出的幻灯片不等式为

$$
\mathbb P_\sigma\left(
\max_{h\in\widetilde{\mathcal H}_{C_X}}|\theta_h|>\rho
\right)
\le2|\mathcal H_{C_X}|e^{-2m\rho^2}.
$$


# 33. 第六步：Massart 引理或尾积分

*幻灯片主线：L10，第 19 页。*

Massart 引理（Massart's lemma）所给出的有限类最大值不等式（finite-class maximal inequality）为

$$
\mathbb E_\sigma
\left[
\max_{h\in\widetilde{\mathcal H}_{C_X}}|\theta_h|
\right]
\le
\frac{4+\sqrt{\log|\mathcal H_{C_X}|}}{\sqrt{2m}}.
$$

同类结果也可以通过尾积分（tail integration）公式

$$
\mathbb EX=\int_0^\infty\mathbb P(X>t)\,dt
$$

与前面的尾概率界得到。由于 $|C_X|\le2m$，

$$
|\mathcal H_{C_X}|\le\tau_{\mathcal H}(2m).
$$

合并全部六步可得

$$
\boxed{
\mathbb E_S
\sup_{h\in\mathcal H}|L_{\mathbb P}(h)-L_S(h)|
\le
\frac{4+\sqrt{\log\tau_{\mathcal H}(2m)}}{\sqrt{2m}}
}.
$$

最后用 Markov 不等式即可完成高概率定理。再结合 Sauer 引理，就证明了统计学习基本定理中的 $6\Rightarrow1$。幻灯片为这个“由尾界到期望”的步骤引用了 Massart 引理与 [SSS] 附录 A.4（Appendix A.4）。

## 33.1 补充：Rademacher 复杂度与更精细的视角

对于实值函数类 $\mathcal F$，经验 Rademacher 复杂度（empirical Rademacher complexity）为

$$
\widehat{\mathfrak R}_S(\mathcal F)
=\mathbb E_\sigma
\left[
\sup_{f\in\mathcal F}
\frac1m\sum_{i=1}^m\sigma_i f(X_i)
\right].
$$

它衡量函数类与随机噪声（random noise）对齐的能力。L10 的证明是这一方法在有限标记上的版本。对称化之后再应用有界差分集中（bounded-difference concentration），可以得到更标准的置信度依赖：

$$
\sup_{h\in\mathcal H}|L_{\mathbb P}(h)-L_S(h)|
=O\left(
\sqrt{\frac{d+\log(1/\delta)}{m}}
\right),
$$

具体形式仍取决于所使用的定理与常数。


# 34. 通过算法稳定性解释泛化

*幻灯片主线：L10，第 20--21 页。*

一致收敛控制整个假设类，因此可能过于悲观。算法稳定性转而研究：当输入数据受到轻微扰动（perturbation）时，一个具体算法会改变多少。

幻灯片列出四种扰动：

1. 对一个样本点重新采样（resample one point）；
2. 留出一个样本点（leave one point out）；
3. 任意污染一个样本点（corrupt one point）；
4. 向算法本身加入噪声。

如果输出对这些改变不敏感，那么算法就不太可能记忆某个特殊的训练点，因此泛化间隙（generalization gap）应当较小。

**幻灯片“Explore”问题。** 比较这些扰动如何影响稳定性与泛化。重新采样衡量替换稳定性（replacement stability），并与下面的混合样本证明相匹配；留一法衡量删除稳定性（deletion stability）；任意污染要求对抗鲁棒性（adversarial robustness）或最坏情形鲁棒性（worst-case robustness）；算法内部噪声则使稳定性成为随机化学习器（randomized learner）的性质。每一种扰动都可以在期望意义或最坏情形下研究，因此所得保证的强弱不同。


# 35. 稳定性设定与混合样本

*幻灯片主线：L10，第 22 页。*

令

$$
S=(z_1,\ldots,z_n),
\qquad
S'=(z_1',\ldots,z_n')
$$

为两个独立样本。把算法视为映射

$$
A:\mathcal Z^n\to\mathcal H.
$$

幻灯片写成 $A:\mathbb P^n\to\mathcal H$；上面的标准类型写法把样本空间（sample space）与其概率分布区分开。

对于每个 $i$，构造混合样本（hybrid sample）

$$
S^i=(z_1,\ldots,z_{i-1},z_i',z_{i+1},\ldots,z_n).
$$

这与前面的幽灵样本思想相同，但现在用于比较算法输出。


# 36. 平均稳定性等于期望泛化间隙

*幻灯片主线：L10，第 23--24 页。*

对于 $z=(x,y)$，幻灯片首先使用简记（shorthand）

$$
\ell(f,z)=\ell(f(x),y).
$$

然后把下面的概念标为**平均稳定性（依赖数据，average stability, data dependent）**，其定义为

$$
\Delta(A)
=\mathbb E_{S,S'}
\left[
\frac1n\sum_{i=1}^n
\left(
\ell(A(S),z_i')-
\ell(A(S^i),z_i')
\right)
\right].
$$

这里，$z_i'$ 对 $A(S)$ 而言是未见样本，但对 $A(S^i)$ 而言是已见样本。因此，$\Delta(A)$ 既是平均的单点敏感度（one-point sensitivity），也是平均的“已见与未见损失差”。

命题给出如下精确恒等式（exact identity）：

$$
\boxed{
\mathbb E_S
\left[
L_{\mathbb P}(A(S))-L_S(A(S))
\right]
=\Delta(A)
}.
$$

证明如下：

$$
\begin{aligned}
&\mathbb E[L_{\mathbb P}(A(S))-L_S(A(S))]\\
&=
\mathbb E\left[\frac1n\sum_i\ell(A(S),z_i')\right]
-
\mathbb E\left[\frac1n\sum_i\ell(A(S),z_i)\right].
\end{aligned}
$$

$z_i$ 与 $z_i'$ 的可交换性给出

$$
\mathbb E[\ell(A(S),z_i)]
=\mathbb E[\ell(A(S^i),z_i')].
$$

代入后即得 $\Delta(A)$。幻灯片把左侧称为“期望超额风险（expected excess risk）”，并紧接着在括号中写“泛化间隙”。严格地说，超额风险通常指 $L_{\mathbb P}(h)-\inf_{g\in\mathcal H}L_{\mathbb P}(g)$；这里的恒等式讨论的是期望泛化间隙（expected generalization gap）。


# 37. 一致稳定性

*幻灯片主线：L10，第 25 页。*

令 $d_H(S,S')$ 表示样本之间的 Hamming 距离（Hamming distance）。一致稳定性（uniform stability）是只替换一个样本时，损失在最坏情形下的变化：

$$
\Delta_{\sup}(A)
=
\sup_{\substack{S,S'\\d_H(S,S')=1}}
\sup_{z}
\left|
\ell(A(S),z)-\ell(A(S'),z)
\right|.
$$

幻灯片把第二个上确界写成 $z\sim\mathbb P$；标准理解是对分布支撑集（support）中的测试点取上确界。由于

$$
|\Delta(A)|\le\Delta_{\sup}(A),
$$

一致稳定性也能控制期望泛化间隙。

**补充。** 正则化（regularization）所带来的强凸性（strong convexity），常常会使正则化 ERM（regularized ERM）具有一致稳定性。这是一种依赖算法（algorithm-dependent）的解释，可能比对每个 $h\in\mathcal H$ 都成立的最坏情形界精确得多。


# 38. SGD 又如何？

*幻灯片主线：L10，第 26 页。*

幻灯片首先指出，SGD 是使用最广泛的训练方法之一；更准确地说，实际训练通常使用带动量的 SGD、Adam 或 AdamW。随后，本段学习理论课程以如下开放问题（open question）结束：

- 随机梯度下降（stochastic gradient descent, SGD）是否满足一致稳定性？
- SGD 会选择哪一类解？
- 过参数化（overparameterization）与损失曲面（loss landscape）如何影响这些解？
- 对带动量的 SGD（SGD with momentum）、Adam 或 AdamW，又会发生什么变化？

这些问题把泛化与优化以及隐式偏置联系起来。因此，课程的下一部分转向优化。


# 39. 幻灯片明确列出的参考资料

*幻灯片主线：L10，第 27 页。*

对于 PAC 学习、不可知 PAC 学习、一致收敛与 VC 维，幻灯片推荐 Shalev-Shwartz 与 Ben-David 教材的第 3、4、6 章，并推荐第 26--28 章中的技术细节。

对于算法稳定性，幻灯片推荐 Hardt 与 Recht（2021）的 [*Patterns, Predictions, and Actions*](https://mlstory.org/) 第 6 章。

幻灯片还展示了：

- Yiding Jiang、Behnam Neyshabur、Hossein Mobahi、Dilip Krishnan 与 Samy Bengio 的论文 *Fantastic Generalization Measures and Where to Find Them*，ICLR 2020；
- [PGDL 2020 网站](https://sites.google.com/view/pgdl2020)；
- NeurIPS 2020 竞赛说明（competition note）。

本讲义使用的本地补充教材是 [Learning Theory from First Principles](../slides/ltfp_book.pdf) 与 [Understanding Machine Learning - Solution Manual](../slides/MLbookSol.pdf)。


# 40. 幻灯片内容覆盖索引

本索引明确标出动画逐步显现页面以及以视觉内容为主的页面如何被纳入讲义。

- **L7-8 第 1--3 页：** 标题、此前课程主题、学习理论的动机——第 1、40 节。
- **L7-8 第 4--11 页：** 记号、风险、损失、ERM、归纳偏置与目标保证——第 2--4 节。
- **L7-8 第 12--19 页：** 误差分解、讨论问题、经典 U 形曲线与双降——第 5--6 节。
- **L7-8 第 20--23 页：** 可实现性及其练习、有限假设类假设与剩余的泛化问题——第 7 节。
- **L7-8 第 24--30 页：** 有限类定理、误导性样本的完整证明、PAC 定义、$\varepsilon/\delta$、计算方面的说明与 Valiant——第 8--10 节。
- **L7-8 第 31--35 页：** 无限假设类、带具体常数与练习的 NFL、不可知目标及对框架的批评——第 10--12 节。
- **L7-8 第 36--38 页：** 代表性样本、完整的 ERM 不等式链与一致收敛——第 13 节。
- **L9 第 1--9 页：** PAC、不可知 PAC、一致收敛回顾以及向 VC 维的过渡——第 10--14 节。
- **L9 第 10--17 页：** 超平面动机、增长函数定义、$\tau(2)=4$、$\tau(3)=6$、二分现象、速率启发式、未见点上 $1/2$ 的直觉与总结——第 15--17 节。
- **L9 第 18--21 页：** 打散、VC 维、平面线性分类器、证明模板、阈值、有限假设类练习与阅读要求——第 18--21 节。
- **L9 第 22--24 页：** 六条件等价的基本定理、回顾、其他复杂度与 PAC-Bayes——第 22--23 节。
- **L10 第 1--3 页：** 基本定理、定量速率、$6\Rightarrow1$、增长函数差异定理与推论练习——第 22、24 节。
- **L10 第 4--7 页：** 限制、两个打散定义、增长函数、一维标记图、Sauer 引理、例子、历史与练习——第 25--26 节。
- **L10 第 8--19 页：** 期望差异以及直到 Markov 不等式的全部六个证明步骤——第 27--33 节。
- **L10 第 20--25 页：** 稳定性分隔页、扰动类型、混合样本、平均稳定性恒等式与一致稳定性——第 34--37 节。
- **L10 第 26--27 页：** SGD 问题、向优化部分的过渡以及所有展示的参考资料——第 38--39 节。


# 41. 最终证明路线图

整套理论可以通过四条路线记忆。

**有限且可实现的假设类：**

$$
\text{坏假设拟合全部 }n\text{ 个样本}
\Longrightarrow e^{-n\varepsilon}
\xRightarrow{\text{并集界}}
|\mathcal H|e^{-n\varepsilon}.
$$

**VC 路线：**

$$
\operatorname{VCdim}(\mathcal H)=d
\xRightarrow{\text{Sauer 引理}}
\tau_{\mathcal H}(m)\le(em/d)^d
\Longrightarrow\text{一致收敛}
\Longrightarrow\text{不可知 ERM}.
$$

**Rademacher 路线：**

$$
\text{幽灵样本}
\Longrightarrow\text{对称化}
\Longrightarrow\text{随机符号上确界（random-sign supremum）}
\Longrightarrow\text{泛化界（generalization bound）}.
$$

**稳定性路线：**

$$
S\text{ 中的一个点发生改变}
\Longrightarrow A(S)\text{ 只发生很小改变}
\Longrightarrow
\mathbb E[L_{\mathbb P}(A(S))-L_S(A(S))]\text{ 很小}.
$$

总之，三份幻灯片给出了两种互补的泛化解释：控制整个假设类的有效容量（effective capacity），或控制特定学习算法的敏感度（sensitivity）。
